# Centralized sequential convex QP on the exact corrected CTM

This notebook optimizes ramp-metering commands against the **exact corrected benchmark plant** (`Benchmark_calculation` v3). It does not re-implement the CTM: it loads the exact one-step functions from the corrected benchmark notebook, linearizes the exact transition around a physically valid nominal trajectory, solves a trust-region convex QP, and re-linearizes (sequential convexification).

**Optimization guarantees in this version:** the loop starts from the feasible fully-open benchmark, every candidate is line-searched and accepted only if its **exact CTM replay** strictly improves the objective (monotone descent, asserted), the accepted-release-to-command step is an explicitly verified trajectory-preserving projection, every linearization is anchor-asserted to its exact nominal trajectory, the trust region adapts to measured model mismatch, and the final policy can never be worse than the benchmark initialization (asserted). Non-converged terminations are labeled as such.

Rules enforced here:
1. `commanded_release_series` is a meter command, never a PeMS observation; the fully-open benchmark command is non-binding by construction.
2. The QP is a **local surrogate**; the only number used for ranking is the **exact CTM replay** objective of the final commands.
3. Every objective component is vehicle-minutes, matching the corrected benchmark decomposition, and the replay is checked against all physical invariants (sending, receiving, merge, and four mass ledgers).


In [4]:
# 1. Load the corrected benchmark and exact CTM functions.

from pathlib import Path
import time

import numpy as np
import pandas as pd
import cvxpy as cp
from IPython.display import display
from IPython.utils.capture import (
    capture_output
)


BENCHMARK_NOTEBOOK = Path(
    "Benchmark_calculation.ipynb"
)

if not BENCHMARK_NOTEBOOK.exists():
    raise FileNotFoundError(
        f"{BENCHMARK_NOTEBOOK} was not found. "
        "Set BENCHMARK_NOTEBOOK to the exact corrected "
        "benchmark notebook filename."
    )

# Always execute the exact benchmark used
# for this QP run. Output is suppressed,
# but exceptions are not suppressed.
with capture_output():
    get_ipython().run_line_magic(
        "run",
        f'"{BENCHMARK_NOTEBOOK}"'
    )

if (
    "shared_benchmark_inputs"
    not in globals()
):
    raise RuntimeError(
        "The corrected benchmark did not define "
        "shared_benchmark_inputs."
    )

required_exact_functions = [
    "ramp_requested_release_one_step",
    "ramp_next_queue_after_actual_release",
    "ctm_15sec_step",
    "simulate_state_based_benchmark_480_steps",
    "compute_mainline_mass_residual",
    "compute_upstream_boundary_mass_residual",
    "compute_external_entry_mass_residual",
    "compute_max_sending_violation",
    "compute_max_receiving_violation",
    "compute_max_merge_violation",
]

missing_functions = [
    name
    for name in required_exact_functions
    if name not in globals()
]

if missing_functions:
    raise RuntimeError(
        "Missing corrected benchmark functions: "
        f"{missing_functions}"
    )

S = shared_benchmark_inputs


cells = [
    f"Cell {index}"
    for index in range(1, 10)
]

ramps = list(
    S["ramp_ids"]
)

T = int(
    S["num_steps"]
)

dt = float(
    S["delta_t"]
)

nX = len(cells)
nR = len(ramps)
nE = len(cells)


if T != 480:
    raise ValueError(
        f"Expected 480 steps, received {T}."
    )


x0 = np.array(
    [
        float(S["mainline_initial_state"][cell])
        for cell in cells
    ],
    dtype=float
)

W0 = np.array(
    [
        float(S["ramp_queue_0"][ramp])
        + float(S["ramp_overflow_queue_0"][ramp])
        for ramp in ramps
    ],
    dtype=float
)

U0 = 0.0

E0 = np.array(
    [
        float(S["external_entry_queue_0"][cell])
        for cell in cells
    ],
    dtype=float
)


a = np.array(
    [
        [
            float(
                S["ramp_arrival_series"][ramp][step]
            )
            for step in range(T)
        ]
        for ramp in ramps
    ],
    dtype=float
)

boundary_arrival = np.array(
    [
        float(S["q_in_boundary_series"][step])
        for step in range(T)
    ],
    dtype=float
)

Xmax = np.array(
    [
        float(S["physical_capacity"][cell])
        for cell in cells
    ],
    dtype=float
)

Xsafe = np.array(
    [
        float(S["safe_threshold_capacity"][cell])
        for cell in cells
    ],
    dtype=float
)

Rmax = np.array(
    [
        float(S["ramp_max_queue_by_u"][ramp])
        for ramp in ramps
    ],
    dtype=float
)


AW = float(
    S["AW"]
)

gamma = float(
    S["gamma"]
)

lambda_safe = float(
    S["lambda_safe"]
)

lambda_spillback = float(
    S["lambda_spillback"]
)


benchmark_totals = dict(
    S["official_totals"]
)

benchmark_history = S[
    "official_benchmark_history"
]


print(
    "Loaded corrected benchmark:",
    S["benchmark_policy_name"]
)

print(
    "Steps:",
    T,
    "| benchmark objective:",
    benchmark_totals["raw_objective"]
)


Loaded corrected benchmark: Fully open / no control
Steps: 480 | benchmark objective: 88338.75071339465


In [5]:
# 2. Exact state representation used for local linearization.
#
# State ordering:
#   0:9    mainline inventories x
#   9:13   total ramp queues W = R + B
#   13     upstream boundary queue U
#   14:23  external-entry queues E

X_SLICE = slice(0, 9)
W_SLICE = slice(9, 13)
U_INDEX = 13
E_SLICE = slice(14, 23)

STATE_SIZE = 23


def pack_state(
    x_value,
    W_value,
    U_value,
    E_value
):
    return np.concatenate([
        np.asarray(
            x_value,
            dtype=float
        ),

        np.asarray(
            W_value,
            dtype=float
        ),

        np.array(
            [float(U_value)],
            dtype=float
        ),

        np.asarray(
            E_value,
            dtype=float
        ),
    ])


def unpack_state(state_vector):
    state_vector = np.asarray(
        state_vector,
        dtype=float
    )

    if state_vector.shape != (STATE_SIZE,):
        raise ValueError(
            "Invalid exact-state vector shape."
        )

    x_value = state_vector[X_SLICE]
    W_value = state_vector[W_SLICE]
    U_value = float(
        state_vector[U_INDEX]
    )
    E_value = state_vector[E_SLICE]

    return (
        x_value,
        W_value,
        U_value,
        E_value
    )


initial_state_vector = pack_state(
    x_value=x0,
    W_value=W0,
    U_value=U0,
    E_value=E0
)


def exact_transition(
    state_vector,
    command_vector,
    step
):
    x_value, W_value, U_value, E_value = (
        unpack_state(state_vector)
    )

    command_vector = np.asarray(
        command_vector,
        dtype=float
    )

    if command_vector.shape != (nR,):
        raise ValueError(
            "Invalid ramp-command vector shape."
        )

    x_current = {
        cell: float(x_value[index])
        for index, cell in enumerate(cells)
    }

    # Split total ramp queue into physical queue R
    # and overflow queue B exactly as the benchmark does.
    R_current = {
        ramp: min(
            float(W_value[index]),
            float(Rmax[index])
        )
        for index, ramp in enumerate(ramps)
    }

    B_current = {
        ramp: max(
            0.0,
            float(W_value[index])
            - float(Rmax[index])
        )
        for index, ramp in enumerate(ramps)
    }

    ramp_arrival_step = {
        ramp: float(
            S["ramp_arrival_series"][ramp][step]
        )
        for ramp in ramps
    }

    commanded_release_step = {
        ramp: max(
            0.0,
            float(command_vector[index])
        )
        for index, ramp in enumerate(ramps)
    }

    (
        available_demand_step,
        requested_release_step,
    ) = ramp_requested_release_one_step(
        R_current=R_current,
        B_current=B_current,
        ramp_arrival_step=ramp_arrival_step,
        commanded_release_step=commanded_release_step
    )

    external_entry_queue = {
        cell: float(E_value[index])
        for index, cell in enumerate(cells)
    }

    (
        x_next,
        q_out,
        q_in,
        sending,
        receiving,
        receiving_after_external,
        actual_f_out,
        split_exit_flow,
        actual_fixed_f_out,
        actual_release_step,
        controllable_onramp_in_step,
        upstream_boundary_queue_next,
        upstream_boundary_delay,
        merge_diagnostics,
        accepted_external,
        external_entry_queue_next,
    ) = ctm_15sec_step(
        x_current=x_current,

        q_in_boundary_step=float(
            S["q_in_boundary_series"][step]
        ),

        external_inflow_step=(
            S["external_inflow_series"][step]
        ),

        fixed_outflow_step=(
            S["fixed_outflow_series"][step]
        ),

        requested_release_step=(
            requested_release_step
        ),

        inflow_capacity=(
            S["inflow_capacity"]
        ),

        outflow_capacity=(
            S["outflow_capacity"]
        ),

        physical_capacity=(
            S["physical_capacity"]
        ),

        movement_factor_by_cell=(
            S["movement_factor_by_cell"]
        ),

        wave_speed_ratio_by_cell=(
            S["wave_speed_ratio_by_cell"]
        ),

        upstream_boundary_queue=U_value,

        exit_split_by_cell=(
            S["exit_split_by_cell"]
        ),

        merge_priority=float(
            S["MERGE_PRIORITY"]
        ),

        external_entry_queue=(
            external_entry_queue
        )
    )

    (
        R_next,
        B_next,
        spillback_by_ramp,
    ) = ramp_next_queue_after_actual_release(
        available_demand_step=(
            available_demand_step
        ),

        actual_release_step=(
            actual_release_step
        ),

        ramp_max_queue_by_u=(
            S["ramp_max_queue_by_u"]
        )
    )

    next_x = np.array(
        [
            float(x_next[cell])
            for cell in cells
        ],
        dtype=float
    )

    next_W = np.array(
        [
            float(R_next[ramp])
            + float(B_next[ramp])
            for ramp in ramps
        ],
        dtype=float
    )

    next_E = np.array(
        [
            float(
                external_entry_queue_next[cell]
            )
            for cell in cells
        ],
        dtype=float
    )

    next_state = pack_state(
        x_value=next_x,
        W_value=next_W,
        U_value=upstream_boundary_queue_next,
        E_value=next_E
    )

    actual_release = np.array(
        [
            float(actual_release_step[ramp])
            for ramp in ramps
        ],
        dtype=float
    )

    requested_release = np.array(
        [
            float(requested_release_step[ramp])
            for ramp in ramps
        ],
        dtype=float
    )

    return {
        "state_next": next_state,
        "actual_release": actual_release,
        "requested_release": requested_release,
        "sending": sending,
        "receiving": receiving,
    }


def rollout_exact_commands(
    command_matrix
):
    command_matrix = np.asarray(
        command_matrix,
        dtype=float
    )

    if command_matrix.shape != (
        nR,
        T
    ):
        raise ValueError(
            f"Expected command shape {(nR, T)}, "
            f"received {command_matrix.shape}."
        )

    states = np.zeros(
        (STATE_SIZE, T + 1),
        dtype=float
    )

    commanded_releases = np.zeros(
        (nR, T),
        dtype=float
    )

    requested_releases = np.zeros(
        (nR, T),
        dtype=float
    )

    actual_releases = np.zeros(
        (nR, T),
        dtype=float
    )

    states[:, 0] = (
        initial_state_vector
    )

    for step in range(T):
        commanded_releases[
            :,
            step
        ] = np.maximum(
            command_matrix[:, step],
            0.0
        )

        result = exact_transition(
            state_vector=(
                states[:, step]
            ),
            command_vector=(
                commanded_releases[
                    :,
                    step
                ]
            ),
            step=step
        )

        states[:, step + 1] = (
            result["state_next"]
        )

        requested_releases[
            :,
            step
        ] = result[
            "requested_release"
        ]

        actual_releases[
            :,
            step
        ] = result[
            "actual_release"
        ]

        if np.any(
            actual_releases[:, step]
            - requested_releases[:, step]
            > 1e-9
        ):
            raise AssertionError(
                "Actual release exceeded request."
            )

        if np.any(
            requested_releases[:, step]
            - commanded_releases[:, step]
            > 1e-9
        ):
            raise AssertionError(
                "Requested release exceeded command."
            )

    return (
        states,
        commanded_releases,
        requested_releases,
        actual_releases
    )


In [6]:
# 3. Initial nominal trajectory.
#
# Using the benchmark's accepted release as the nominal command
# reproduces the same exact trajectory without using a meaningless
# command thousands of vehicles above current demand.

nominal_commands = np.array(
    [
        [
            float(
                benchmark_history[
                    "actual_release"
                ][step][ramp]
            )
            for step in range(T)
        ]
        for ramp in ramps
    ],
    dtype=float
)


(
    nominal_states,
    nominal_commanded_releases,
    nominal_requested_releases,
    nominal_actual_releases,
) = rollout_exact_commands(
    nominal_commands
)


official_state_matrix = np.zeros(
    (STATE_SIZE, T + 1),
    dtype=float
)

official_state_matrix[:, 0] = (
    initial_state_vector
)


for step in range(T):
    official_x = np.array(
        [
            float(
                benchmark_history["x"][step][cell]
            )
            for cell in cells
        ],
        dtype=float
    )

    official_W = np.array(
        [
            float(
                benchmark_history["R"][step][ramp]
            )
            + float(
                benchmark_history["B"][step][ramp]
            )
            for ramp in ramps
        ],
        dtype=float
    )

    official_U = float(
        benchmark_history[
            "upstream_boundary_queue"
        ][step]
    )

    official_E = np.array(
        [
            float(
                benchmark_history[
                    "external_entry_queue"
                ][step][cell]
            )
            for cell in cells
        ],
        dtype=float
    )

    official_state_matrix[:, step + 1] = (
        pack_state(
            x_value=official_x,
            W_value=official_W,
            U_value=official_U,
            E_value=official_E
        )
    )


initial_nominal_error = float(
    np.max(
        np.abs(
            nominal_states
            - official_state_matrix
        )
    )
)


print(
    "Maximum initial nominal-state error:",
    initial_nominal_error
)


if initial_nominal_error > 1e-8:
    raise AssertionError(
        "The exact nominal rollout does not reproduce "
        "the corrected benchmark."
    )


Maximum initial nominal-state error: 5.329070518200751e-15


In [7]:
# 4. Linearize the exact one-step transition around
# a physically valid exact trajectory.

RELATIVE_DIFFERENCE_STEP = 1e-5
ABSOLUTE_DIFFERENCE_STEP = 1e-6


state_lower_bound = np.zeros(
    STATE_SIZE,
    dtype=float
)

state_upper_bound = np.full(
    STATE_SIZE,
    np.inf,
    dtype=float
)

state_upper_bound[X_SLICE] = Xmax


def linearize_exact_trajectory(
    nominal_states,
    nominal_commands
):
    nominal_states = np.asarray(
        nominal_states,
        dtype=float
    )

    nominal_commands = np.asarray(
        nominal_commands,
        dtype=float
    )

    if nominal_states.shape != (
        STATE_SIZE,
        T + 1
    ):
        raise ValueError(
            "Invalid nominal-state shape."
        )

    if nominal_commands.shape != (
        nR,
        T
    ):
        raise ValueError(
            "Invalid nominal-command shape."
        )

    A_matrices = []
    B_matrices = []
    c_vectors = []

    for step in range(T):
        state_0 = nominal_states[
            :,
            step
        ].copy()

        command_0 = nominal_commands[
            :,
            step
        ].copy()

        base_result = exact_transition(
            state_vector=state_0,
            command_vector=command_0,
            step=step
        )

        next_state_0 = base_result[
            "state_next"
        ]

        linearization_anchor_error = float(
            np.max(
                np.abs(
                    next_state_0
                    - nominal_states[
                        :,
                        step + 1
                    ]
                )
            )
        )

        if linearization_anchor_error > 1e-8:
            raise AssertionError(
                "Nominal command/state pair is not an "
                "exact CTM trajectory at step "
                f"{step}. Error = "
                f"{linearization_anchor_error}"
            )

        A_step = np.zeros(
            (STATE_SIZE, STATE_SIZE),
            dtype=float
        )

        B_step = np.zeros(
            (STATE_SIZE, nR),
            dtype=float
        )

        # State Jacobian.
        for state_index in range(
            STATE_SIZE
        ):
            epsilon = max(
                ABSOLUTE_DIFFERENCE_STEP,
                RELATIVE_DIFFERENCE_STEP
                * max(
                    1.0,
                    abs(state_0[state_index])
                )
            )

            lower_value = max(
                state_lower_bound[state_index],
                state_0[state_index]
                - epsilon
            )

            if np.isfinite(
                state_upper_bound[state_index]
            ):
                upper_value = min(
                    state_upper_bound[state_index],
                    state_0[state_index]
                    + epsilon
                )
            else:
                upper_value = (
                    state_0[state_index]
                    + epsilon
                )

            span = (
                upper_value
                - lower_value
            )

            if span <= 1e-14:
                continue

            state_lower = state_0.copy()
            state_upper = state_0.copy()

            state_lower[state_index] = (
                lower_value
            )

            state_upper[state_index] = (
                upper_value
            )

            next_lower = exact_transition(
                state_vector=state_lower,
                command_vector=command_0,
                step=step
            )["state_next"]

            next_upper = exact_transition(
                state_vector=state_upper,
                command_vector=command_0,
                step=step
            )["state_next"]

            A_step[:, state_index] = (
                next_upper
                - next_lower
            ) / span

        # Command Jacobian.
        available_nominal = (
            state_0[W_SLICE]
            + a[:, step]
        )

        for ramp_index in range(nR):
            epsilon = max(
                ABSOLUTE_DIFFERENCE_STEP,
                RELATIVE_DIFFERENCE_STEP
                * max(
                    1.0,
                    abs(command_0[ramp_index])
                )
            )

            lower_value = max(
                0.0,
                command_0[ramp_index]
                - epsilon
            )

            upper_value = min(
                available_nominal[ramp_index],
                command_0[ramp_index]
                + epsilon
            )

            span = (
                upper_value
                - lower_value
            )

            if span <= 1e-14:
                continue

            command_lower = command_0.copy()
            command_upper = command_0.copy()

            command_lower[ramp_index] = (
                lower_value
            )

            command_upper[ramp_index] = (
                upper_value
            )

            next_lower = exact_transition(
                state_vector=state_0,
                command_vector=command_lower,
                step=step
            )["state_next"]

            next_upper = exact_transition(
                state_vector=state_0,
                command_vector=command_upper,
                step=step
            )["state_next"]

            B_step[:, ramp_index] = (
                next_upper
                - next_lower
            ) / span

        c_step = (
            next_state_0
            - A_step @ state_0
            - B_step @ command_0
        )

        A_matrices.append(A_step)
        B_matrices.append(B_step)
        c_vectors.append(c_step)

    return (
        A_matrices,
        B_matrices,
        c_vectors
    )


In [8]:
# 5. Build and solve one local convex QP.

STATE_TRUST_FRACTION = 0.25
QUEUE_TRUST_FRACTION = 0.75
QUEUE_TRUST_FLOOR = 10.0
BOUNDARY_QUEUE_TRUST = 50.0
EXTERNAL_QUEUE_TRUST = 50.0
CONTROL_TRUST_PER_STEP = 2.0

MAXIMUM_CONSTRAINT_VIOLATION = 1e-5


def build_and_solve_local_qp(
    nominal_states,
    nominal_commands,
    trust_scale
):
    trust_scale = float(
        trust_scale
    )

    if trust_scale <= 0.0:
        raise ValueError(
            "trust_scale must be positive."
        )

    (
        A_matrices,
        B_matrices,
        c_vectors,
    ) = linearize_exact_trajectory(
        nominal_states=nominal_states,
        nominal_commands=nominal_commands
    )

    state = cp.Variable(
        (STATE_SIZE, T + 1)
    )

    command = cp.Variable(
        (nR, T)
    )

    safe_excess = cp.Variable(
        (nX, T),
        nonneg=True
    )

    spillback_state = cp.Variable(
        (nR, T + 1),
        nonneg=True
    )

    x = state[
        X_SLICE,
        :
    ]

    W = state[
        W_SLICE,
        :
    ]

    U = state[
        U_INDEX,
        :
    ]

    E = state[
        E_SLICE,
        :
    ]


    state_trust = np.zeros(
        (STATE_SIZE, T + 1),
        dtype=float
    )

    state_trust[X_SLICE, :] = (
        trust_scale
        * (
            STATE_TRUST_FRACTION
            * Xmax[:, None]
            + 1.0
        )
    )

    state_trust[W_SLICE, :] = (
        trust_scale
        * (
            QUEUE_TRUST_FRACTION
            * Rmax[:, None]
            + QUEUE_TRUST_FLOOR
        )
    )

    state_trust[U_INDEX, :] = (
        trust_scale
        * BOUNDARY_QUEUE_TRUST
    )

    state_trust[E_SLICE, :] = (
        trust_scale
        * EXTERNAL_QUEUE_TRUST
    )

    command_trust = (
        trust_scale
        * CONTROL_TRUST_PER_STEP
        * np.ones(
            (nR, T),
            dtype=float
        )
    )


    constraints = [
        state[:, 0]
        == initial_state_vector,

        state >= 0.0,

        x <= Xmax[:, None],

        command >= 0.0,

        state - nominal_states
        <= state_trust,

        nominal_states - state
        <= state_trust,

        command - nominal_commands
        <= command_trust,

        nominal_commands - command
        <= command_trust,

        spillback_state
        >= W - Rmax[:, None],

        safe_excess
        >= (
            x[:, :T]
            + x[:, 1:]
        ) / 2.0 - Xsafe[:, None],
    ]


    for step in range(T):
        # The command cannot release vehicles that do not exist.
        constraints.append(
            command[:, step]
            <= W[:, step]
            + a[:, step]
        )

        # Equality-constrained local CTM dynamics.
        constraints.append(
            state[:, step + 1]
            == (
                A_matrices[step]
                @ state[:, step]
                + B_matrices[step]
                @ command[:, step]
                + c_vectors[step]
            )
        )


    mainline_tts_expression = (
        dt
        * cp.sum(
            (
                x[:, :T]
                + x[:, 1:]
            ) / 2.0
        )
    )

    ramp_tts_expression = (
        dt
        * cp.sum(
            (
                W[:, :T]
                + W[:, 1:]
            ) / 2.0
        )
    )

    boundary_tts_expression = (
        dt
        * cp.sum(
            (
                U[:T]
                + U[1:]
            ) / 2.0
        )
    )

    external_tts_expression = (
        dt
        * cp.sum(
            (
                E[:, :T]
                + E[:, 1:]
            ) / 2.0
        )
    )

    safe_exposure_expression = (
        dt
        * cp.sum(
            safe_excess
        )
    )

    spillback_exposure_expression = (
        dt
        * cp.sum(
            (
                spillback_state[:, :T]
                + spillback_state[:, 1:]
            ) / 2.0
        )
    )


    average_queue = (
        W[:, :T]
        + W[:, 1:]
    ) / 2.0

    normalized_average_queue = cp.multiply(
        1.0 / Rmax[:, None],
        average_queue
    )

    mean_normalized_queue = cp.reshape(
        cp.sum(
            normalized_average_queue,
            axis=0
        ) / nR,
        (1, T),
        order="C"
    )

    mean_normalized_queue_matrix = (
        np.ones((nR, 1))
        @ mean_normalized_queue
    )

    fairness_reference_queue = float(
        np.mean(Rmax)
    )

    fairness_expression = (
        gamma
        * dt
        * fairness_reference_queue
        * cp.sum_squares(
            normalized_average_queue
            - mean_normalized_queue_matrix
        )
    )


    total_objective_expression = (
        AW
        * mainline_tts_expression

        + ramp_tts_expression

        + boundary_tts_expression

        + external_tts_expression

        + fairness_expression

        + lambda_safe
        * safe_exposure_expression

        + lambda_spillback
        * spillback_exposure_expression
    )


    problem = cp.Problem(
        cp.Minimize(
            total_objective_expression
        ),
        constraints
    )


    if not problem.is_dcp():
        raise AssertionError(
            "The local surrogate is not DCP."
        )

    if not problem.is_qp():
        raise AssertionError(
            "The local surrogate is not a QP."
        )


    available_solvers = set(
        cp.installed_solvers()
    )

    start_time = time.time()

    if "CLARABEL" in available_solvers:
        solver_used = "CLARABEL"

        problem.solve(
            solver=cp.CLARABEL,
            warm_start=True,
            tol_gap_abs=1e-8,
            tol_gap_rel=1e-8,
            tol_feas=1e-8,
            max_iter=1000,
            verbose=False
        )

    elif "OSQP" in available_solvers:
        solver_used = "OSQP"

        problem.solve(
            solver=cp.OSQP,
            warm_start=True,
            eps_abs=1e-7,
            eps_rel=1e-7,
            max_iter=50000,
            polishing=True,
            verbose=False
        )

    else:
        raise RuntimeError(
            "Neither CLARABEL nor OSQP is installed."
        )

    solve_seconds = (
        time.time()
        - start_time
    )


    if problem.status not in {
        "optimal",
        "optimal_inaccurate",
    }:
        raise RuntimeError(
            f"QP solve failed: {problem.status}"
        )


    if (
        state.value is None
        or command.value is None
        or problem.value is None
    ):
        raise RuntimeError(
            "QP returned no numerical solution."
        )


    state_value = np.asarray(
        state.value,
        dtype=float
    )

    command_value = np.asarray(
        command.value,
        dtype=float
    )


    if not np.isfinite(
        float(problem.value)
    ):
        raise RuntimeError(
            "QP returned a nonfinite objective."
        )


    if not np.all(
        np.isfinite(state_value)
    ):
        raise RuntimeError(
            "QP returned nonfinite state values."
        )


    if not np.all(
        np.isfinite(command_value)
    ):
        raise RuntimeError(
            "QP returned nonfinite command values."
        )


    maximum_violation = 0.0

    for constraint_index, constraint in enumerate(
        constraints
    ):
        violation = constraint.violation()

        if violation is None:
            continue

        violation_array = np.asarray(
            violation,
            dtype=float
        )

        if not np.all(
            np.isfinite(violation_array)
        ):
            raise RuntimeError(
                "Constraint "
                f"{constraint_index} returned "
                "a nonfinite violation."
            )

        maximum_violation = max(
            maximum_violation,
            float(
                np.max(
                    np.abs(
                        violation_array
                    )
                )
            )
        )


    if (
        maximum_violation
        > MAXIMUM_CONSTRAINT_VIOLATION
    ):
        raise RuntimeError(
            "Maximum QP constraint violation "
            f"{maximum_violation} exceeds "
            f"{MAXIMUM_CONSTRAINT_VIOLATION}."
        )


    components = {
        "mainline_tts": float(
            mainline_tts_expression.value
        ),

        "ramp_queue_tts": float(
            ramp_tts_expression.value
        ),

        "upstream_boundary_tts": float(
            boundary_tts_expression.value
        ),

        "external_entry_tts": float(
            external_tts_expression.value
        ),

        "fairness_penalty": float(
            fairness_expression.value
        ),

        "safe_exposure": float(
            safe_exposure_expression.value
        ),

        "spillback_exposure": float(
            spillback_exposure_expression.value
        ),

        "weighted_safe_penalty": float(
            lambda_safe
            * safe_exposure_expression.value
        ),

        "weighted_spillback_penalty": float(
            lambda_spillback
            * spillback_exposure_expression.value
        ),

        "raw_objective": float(
            problem.value
        ),
    }

    for component_name, component_value in (
        components.items()
    ):
        if not np.isfinite(
            component_value
        ):
            raise RuntimeError(
                "QP component "
                f"{component_name} is nonfinite."
            )


    return {
        "problem": problem,

        "state": state_value,

        "command": command_value,

        "components": components,

        "status": problem.status,

        "solver": solver_used,

        "solve_seconds": solve_seconds,

        "maximum_constraint_violation": (
            maximum_violation
        ),
    }

In [9]:
# 1. Load the corrected benchmark and exact CTM functions.

from pathlib import Path
import time

import numpy as np
import pandas as pd
import cvxpy as cp
from IPython.display import display
from IPython.utils.capture import (
    capture_output
)


BENCHMARK_NOTEBOOK = Path(
    "Benchmark_calculation.ipynb"
)

if not BENCHMARK_NOTEBOOK.exists():
    raise FileNotFoundError(
        f"{BENCHMARK_NOTEBOOK} was not found. "
        "Set BENCHMARK_NOTEBOOK to the exact corrected "
        "benchmark notebook filename."
    )

# Always execute the exact benchmark used
# for this QP run. Output is suppressed,
# but exceptions are not suppressed.
with capture_output():
    get_ipython().run_line_magic(
        "run",
        f'"{BENCHMARK_NOTEBOOK}"'
    )

if (
    "shared_benchmark_inputs"
    not in globals()
):
    raise RuntimeError(
        "The corrected benchmark did not define "
        "shared_benchmark_inputs."
    )

required_exact_functions = [
    "ramp_requested_release_one_step",
    "ramp_next_queue_after_actual_release",
    "ctm_15sec_step",
    "simulate_state_based_benchmark_480_steps",
    "compute_mainline_mass_residual",
    "compute_upstream_boundary_mass_residual",
    "compute_external_entry_mass_residual",
    "compute_max_sending_violation",
    "compute_max_receiving_violation",
    "compute_max_merge_violation",
]

missing_functions = [
    name
    for name in required_exact_functions
    if name not in globals()
]

if missing_functions:
    raise RuntimeError(
        "Missing corrected benchmark functions: "
        f"{missing_functions}"
    )

S = shared_benchmark_inputs


cells = [
    f"Cell {index}"
    for index in range(1, 10)
]

ramps = list(
    S["ramp_ids"]
)

T = int(
    S["num_steps"]
)

dt = float(
    S["delta_t"]
)

nX = len(cells)
nR = len(ramps)
nE = len(cells)


if T != 480:
    raise ValueError(
        f"Expected 480 steps, received {T}."
    )


x0 = np.array(
    [
        float(S["mainline_initial_state"][cell])
        for cell in cells
    ],
    dtype=float
)

W0 = np.array(
    [
        float(S["ramp_queue_0"][ramp])
        + float(S["ramp_overflow_queue_0"][ramp])
        for ramp in ramps
    ],
    dtype=float
)

U0 = 0.0

E0 = np.array(
    [
        float(S["external_entry_queue_0"][cell])
        for cell in cells
    ],
    dtype=float
)


a = np.array(
    [
        [
            float(
                S["ramp_arrival_series"][ramp][step]
            )
            for step in range(T)
        ]
        for ramp in ramps
    ],
    dtype=float
)

boundary_arrival = np.array(
    [
        float(S["q_in_boundary_series"][step])
        for step in range(T)
    ],
    dtype=float
)

Xmax = np.array(
    [
        float(S["physical_capacity"][cell])
        for cell in cells
    ],
    dtype=float
)

Xsafe = np.array(
    [
        float(S["safe_threshold_capacity"][cell])
        for cell in cells
    ],
    dtype=float
)

Rmax = np.array(
    [
        float(S["ramp_max_queue_by_u"][ramp])
        for ramp in ramps
    ],
    dtype=float
)


AW = float(
    S["AW"]
)

gamma = float(
    S["gamma"]
)

lambda_safe = float(
    S["lambda_safe"]
)

lambda_spillback = float(
    S["lambda_spillback"]
)


benchmark_totals = dict(
    S["official_totals"]
)

benchmark_history = S[
    "official_benchmark_history"
]


print(
    "Loaded corrected benchmark:",
    S["benchmark_policy_name"]
)

print(
    "Steps:",
    T,
    "| benchmark objective:",
    benchmark_totals["raw_objective"]
)


Loaded corrected benchmark: Fully open / no control
Steps: 480 | benchmark objective: 88338.75071339465


In [10]:
# 6. Exact-feasibility projection utilities (QP correction 2).

def history_ramp_matrix(
    history,
    key
):
    return np.array(
        [
            [
                float(
                    history[key][step][ramp]
                )
                for step in range(T)
            ]
            for ramp in ramps
        ],
        dtype=float
    )


def history_state_matrix(
    history
):
    state_matrix = np.zeros(
        (STATE_SIZE, T + 1),
        dtype=float
    )

    state_matrix[:, 0] = (
        initial_state_vector
    )

    for step in range(T):
        x_value = np.array(
            [
                float(
                    history["x"][step][cell]
                )
                for cell in cells
            ],
            dtype=float
        )

        W_value = np.array(
            [
                float(
                    history["R"][step][ramp]
                )
                + float(
                    history["B"][step][ramp]
                )
                for ramp in ramps
            ],
            dtype=float
        )

        U_value = float(
            history[
                "upstream_boundary_queue"
            ][step]
        )

        E_value = np.array(
            [
                float(
                    history[
                        "external_entry_queue"
                    ][step][cell]
                )
                for cell in cells
            ],
            dtype=float
        )

        state_matrix[:, step + 1] = (
            pack_state(
                x_value=x_value,
                W_value=W_value,
                U_value=U_value,
                E_value=E_value
            )
        )

    return state_matrix


EXACT_SIMULATION_ARGUMENTS = {
    "num_steps": T,

    "mainline_initial_state": (
        S["mainline_initial_state"]
    ),

    "ramp_queue_0": (
        S["ramp_queue_0"]
    ),

    "q_in_boundary_series": (
        S["q_in_boundary_series"]
    ),

    "ramp_arrival_series": (
        S["ramp_arrival_series"]
    ),

    "external_inflow_series": (
        S["external_inflow_series"]
    ),

    "fixed_outflow_series": (
        S["fixed_outflow_series"]
    ),

    "inflow_capacity": (
        S["inflow_capacity"]
    ),

    "outflow_capacity": (
        S["outflow_capacity"]
    ),

    "physical_capacity": (
        S["physical_capacity"]
    ),

    "safe_threshold_capacity": (
        S["safe_threshold_capacity"]
    ),

    "ramp_name_map": (
        S["ramp_name_map"]
    ),

    "ramp_max_queue_named": (
        S["ramp_max_queue_named"]
    ),

    "ramp_max_queue_by_u": (
        S["ramp_max_queue_by_u"]
    ),

    "tt_ff_min": (
        S["tt_ff_min"]
    ),

    "delta_t": dt,

    "gamma": gamma,

    "lambda_safe": lambda_safe,

    "lambda_spillback": (
        lambda_spillback
    ),

    "AW": AW,

    "movement_factor_by_cell": (
        S["movement_factor_by_cell"]
    ),

    "wave_speed_ratio_by_cell": (
        S["wave_speed_ratio_by_cell"]
    ),

    "exit_split_by_cell": (
        S["exit_split_by_cell"]
    ),

    "ramp_overflow_queue_0": (
        S["ramp_overflow_queue_0"]
    ),

    "external_entry_queue_0": (
        S["external_entry_queue_0"]
    ),
}


def command_matrix_to_series(
    command_matrix
):
    command_matrix = np.asarray(
        command_matrix,
        dtype=float
    )

    if command_matrix.shape != (
        nR,
        T
    ):
        raise ValueError(
            "Invalid command-matrix shape."
        )

    return {
        ramp: [
            max(
                0.0,
                float(
                    command_matrix[
                        ramp_index,
                        step
                    ]
                )
            )
            for step in range(T)
        ]
        for ramp_index, ramp in enumerate(
            ramps
        )
    }


def simulate_exact_command_matrix(
    command_matrix
):
    command_series = (
        command_matrix_to_series(
            command_matrix
        )
    )

    return (
        simulate_state_based_benchmark_480_steps(
            commanded_release_series=(
                command_series
            ),
            **EXACT_SIMULATION_ARGUMENTS
        )
    )


def project_candidate_to_effective_commands(
    raw_candidate_commands
):
    (
        raw_exact_states,
        raw_commands,
        raw_requests,
        raw_actual_releases,
    ) = rollout_exact_commands(
        raw_candidate_commands
    )

    # Exact-feasibility projection:
    # use the accepted releases as a new,
    # canonical command schedule.
    projected_commands = (
        raw_actual_releases.copy()
    )

    projected_history = (
        simulate_exact_command_matrix(
            projected_commands
        )
    )

    projected_states = (
        history_state_matrix(
            projected_history
        )
    )

    projected_actual_releases = (
        history_ramp_matrix(
            projected_history,
            "actual_release"
        )
    )

    projection_state_error = float(
        np.max(
            np.abs(
                projected_states
                - raw_exact_states
            )
        )
    )

    projection_release_error = float(
        np.max(
            np.abs(
                projected_actual_releases
                - raw_actual_releases
            )
        )
    )

    effective_command_error = float(
        np.max(
            np.abs(
                projected_actual_releases
                - projected_commands
            )
        )
    )

    if projection_state_error > 1e-8:
        raise AssertionError(
            "Accepted-release projection changed "
            "the exact state trajectory."
        )

    if projection_release_error > 1e-8:
        raise AssertionError(
            "Accepted-release projection changed "
            "the exact accepted releases."
        )

    if effective_command_error > 1e-8:
        raise AssertionError(
            "Projected commands did not reproduce "
            "the projected accepted releases."
        )

    return {
        "raw_commands": raw_commands,

        "raw_requests": raw_requests,

        "raw_actual_releases": (
            raw_actual_releases
        ),

        "projected_commands": (
            projected_commands
        ),

        "projected_history": (
            projected_history
        ),

        "projected_states": (
            projected_states
        ),

        "projection_state_error": (
            projection_state_error
        ),

        "projection_release_error": (
            projection_release_error
        ),

        "effective_command_error": (
            effective_command_error
        ),
    }


In [11]:



# 7. Adaptive trust-region sequential convexification with exact acceptance.
import warnings
warnings.simplefilter("error", RuntimeWarning)

def normalized_state_mismatch(
    predicted_states,
    exact_states,
    reference_states
):
    state_scale = np.maximum(
        1.0,
        np.max(
            np.abs(
                reference_states
            ),
            axis=1
        )
    )

    state_scale[X_SLICE] = np.maximum(
        state_scale[X_SLICE],
        Xmax
    )

    state_scale[W_SLICE] = np.maximum(
        state_scale[W_SLICE],
        Rmax
    )

    normalized_error = (
        np.abs(
            predicted_states
            - exact_states
        )
        / state_scale[:, None]
    )

    return float(
        np.max(
            normalized_error
        )
    )


MAXIMUM_QP_ATTEMPTS = 20
MAXIMUM_ACCEPTED_STEPS = 10

INITIAL_TRUST_SCALE = 0.25
MINIMUM_TRUST_SCALE = 1.0 / 128.0
MAXIMUM_TRUST_SCALE = 2.0

BACKTRACKING_ALPHAS = [
    1.0,
    0.5,
    0.25,
    0.125,
    0.0625,
    0.03125,
]

EXACT_OBJECTIVE_TOLERANCE = 1e-7
PREDICTED_REDUCTION_TOLERANCE = 1e-6
NORMALIZED_MODEL_TOLERANCE = 1e-2
PROJECTED_COMMAND_TOLERANCE = 1e-3

# The initial feasible policy is the benchmark.
nominal_history = benchmark_history

nominal_states = history_state_matrix(
    nominal_history
)

# Canonical effective commands that exactly reproduce
# the benchmark trajectory.
nominal_commands = history_ramp_matrix(
    nominal_history,
    "actual_release"
)

nominal_exact_objective = float(
    sum(
        nominal_history[
            "total_objective"
        ]
    )
)

benchmark_exact_objective = (
    nominal_exact_objective
)

# Verify the initial projected-command trajectory.
initial_projection = (
    project_candidate_to_effective_commands(
        nominal_commands
    )
)

initial_projection_state_error = float(
    np.max(
        np.abs(
            initial_projection[
                "projected_states"
            ]
            - nominal_states
        )
    )
)

if initial_projection_state_error > 1e-8:
    raise AssertionError(
        "Initial effective command does not "
        "reproduce the benchmark trajectory."
    )

trust_scale = (
    INITIAL_TRUST_SCALE
)

accepted_steps = 0

termination_status = (
    "maximum_attempts_reached"
)

outer_diagnostics = []
last_qp_result = None

for attempt in range(
    1,
    MAXIMUM_QP_ATTEMPTS + 1
):
    qp_result = build_and_solve_local_qp(
        nominal_states=nominal_states,
        nominal_commands=nominal_commands,
        trust_scale=trust_scale
    )

    last_qp_result = qp_result

    surrogate_objective = float(
        qp_result[
            "components"
        ]["raw_objective"]
    )

    predicted_reduction = (
        nominal_exact_objective
        - surrogate_objective
    )

    # The nominal trajectory is feasible in the local QP.
    # Therefore its optimum should not be materially worse.
    if predicted_reduction < -1e-5:
        raise AssertionError(
            "The local QP objective is worse than "
            "its feasible nominal trajectory. "
            "Check the linearization anchor and "
            "objective consistency."
        )

    if (
        predicted_reduction
        <= PREDICTED_REDUCTION_TOLERANCE
    ):
        termination_status = (
            "local_surrogate_stationary"
        )

        outer_diagnostics.append({
            "attempt": attempt,
            "accepted": False,
            "trust_scale": trust_scale,
            "surrogate_objective": (
                surrogate_objective
            ),
            "predicted_reduction": (
                predicted_reduction
            ),
            "exact_objective": (
                nominal_exact_objective
            ),
            "exact_improvement": 0.0,
            "alpha": 0.0,
            "normalized_model_mismatch": (
                0.0
            ),
            "projected_command_change": (
                0.0
            ),
            "termination_note": (
                termination_status
            ),
        })

        break

    raw_qp_commands = np.asarray(
        qp_result["command"],
        dtype=float
    )

    command_direction = (
        raw_qp_commands
        - nominal_commands
    )

    best_trial = None
    lowest_trial_objective = np.inf

    for alpha in BACKTRACKING_ALPHAS:
        raw_trial_commands = (
            nominal_commands
            + alpha
            * command_direction
        )

        trial_projection = (
            project_candidate_to_effective_commands(
                raw_trial_commands
            )
        )

        trial_history = trial_projection[
            "projected_history"
        ]

        trial_states = trial_projection[
            "projected_states"
        ]

        trial_commands = trial_projection[
            "projected_commands"
        ]

        trial_exact_objective = float(
            sum(
                trial_history[
                    "total_objective"
                ]
            )
        )

        lowest_trial_objective = min(
            lowest_trial_objective,
            trial_exact_objective
        )

        # Since the local dynamics are affine, the convex
        # interpolation of nominal and candidate states is
        # the local model's prediction for this alpha.
        predicted_trial_states = (
            nominal_states
            + alpha
            * (
                qp_result["state"]
                - nominal_states
            )
        )

        normalized_mismatch = (
            normalized_state_mismatch(
                predicted_states=(
                    predicted_trial_states
                ),
                exact_states=(
                    trial_states
                ),
                reference_states=(
                    nominal_states
                )
            )
        )

        projected_command_change = float(
            np.max(
                np.abs(
                    trial_commands
                    - nominal_commands
                )
            )
        )

        exact_improvement = (
            nominal_exact_objective
            - trial_exact_objective
        )

        if (
            exact_improvement
            > EXACT_OBJECTIVE_TOLERANCE
        ):
            if (
                best_trial is None
                or trial_exact_objective
                < best_trial[
                    "exact_objective"
                ]
            ):
                best_trial = {
                    "alpha": alpha,
                    "commands": (
                        trial_commands
                    ),
                    "states": (
                        trial_states
                    ),
                    "history": (
                        trial_history
                    ),
                    "exact_objective": (
                        trial_exact_objective
                    ),
                    "exact_improvement": (
                        exact_improvement
                    ),
                    "normalized_mismatch": (
                        normalized_mismatch
                    ),
                    "command_change": (
                        projected_command_change
                    ),
                    "projection_state_error": (
                        trial_projection[
                            "projection_state_error"
                        ]
                    ),
                    "projection_release_error": (
                        trial_projection[
                            "projection_release_error"
                        ]
                    ),
                }

    if best_trial is None:
        outer_diagnostics.append({
            "attempt": attempt,
            "accepted": False,
            "trust_scale": trust_scale,
            "surrogate_objective": (
                surrogate_objective
            ),
            "predicted_reduction": (
                predicted_reduction
            ),
            "exact_objective": (
                nominal_exact_objective
            ),
            "lowest_trial_objective": (
                lowest_trial_objective
            ),
            "exact_improvement": 0.0,
            "alpha": 0.0,
            "normalized_model_mismatch": (
                np.nan
            ),
            "projected_command_change": (
                np.nan
            ),
            "termination_note": (
                "rejected_exact_replay"
            ),
        })

        trust_scale *= 0.5

        if (
            trust_scale
            < MINIMUM_TRUST_SCALE
        ):
            termination_status = (
                "no_exact_improving_step"
            )
            break

        continue

    previous_exact_objective = (
        nominal_exact_objective
    )

    nominal_commands = (
        best_trial["commands"]
    )

    nominal_states = (
        best_trial["states"]
    )

    nominal_history = (
        best_trial["history"]
    )

    nominal_exact_objective = float(
        best_trial[
            "exact_objective"
        ]
    )

    accepted_steps += 1

    if (
        nominal_exact_objective
        > previous_exact_objective
        + EXACT_OBJECTIVE_TOLERANCE
    ):
        raise AssertionError(
            "Accepted exact objective increased."
        )

    normalized_mismatch = float(
        best_trial[
            "normalized_mismatch"
        ]
    )

    projected_command_change = float(
        best_trial[
            "command_change"
        ]
    )

    # Adaptive trust-region update.
    if (
        normalized_mismatch
        > 2.0
        * NORMALIZED_MODEL_TOLERANCE
    ):
        trust_scale = max(
            MINIMUM_TRUST_SCALE,
            0.5 * trust_scale
        )
    elif (
        normalized_mismatch
        < 0.5
        * NORMALIZED_MODEL_TOLERANCE
        and best_trial["alpha"] == 1.0
    ):
        trust_scale = min(
            MAXIMUM_TRUST_SCALE,
            1.25 * trust_scale
        )

    outer_diagnostics.append({
        "attempt": attempt,
        "accepted": True,
        "trust_scale": trust_scale,
        "surrogate_objective": (
            surrogate_objective
        ),
        "predicted_reduction": (
            predicted_reduction
        ),
        "exact_objective": (
            nominal_exact_objective
        ),
        "exact_improvement": (
            best_trial[
                "exact_improvement"
            ]
        ),
        "alpha": (
            best_trial["alpha"]
        ),
        "normalized_model_mismatch": (
            normalized_mismatch
        ),
        "projected_command_change": (
            projected_command_change
        ),
        "projection_state_error": (
            best_trial[
                "projection_state_error"
            ]
        ),
        "projection_release_error": (
            best_trial[
                "projection_release_error"
            ]
        ),
        "termination_note": (
            "accepted"
        ),
    })

    if (
        projected_command_change
        <= PROJECTED_COMMAND_TOLERANCE
        and normalized_mismatch
        <= NORMALIZED_MODEL_TOLERANCE
    ):
        termination_status = (
            "converged"
        )
        break

    if (
        accepted_steps
        >= MAXIMUM_ACCEPTED_STEPS
    ):
        termination_status = (
            "maximum_accepted_steps_reached"
        )
        break

final_commands = (
    nominal_commands.copy()
)

qp_exact_history = (
    nominal_history
)

final_exact_objective = float(
    nominal_exact_objective
)

# Since the benchmark was the initial feasible point,
# the accepted result cannot be worse.
if (
    final_exact_objective
    > benchmark_exact_objective
    + EXACT_OBJECTIVE_TOLERANCE
):
    raise AssertionError(
        "Final exact QP policy is worse than "
        "the feasible benchmark initialization."
    )

outer_diagnostics_df = pd.DataFrame(
    outer_diagnostics
)

display(
    outer_diagnostics_df.round(8)
)

print(
    "Sequential-QP termination status:",
    termination_status
)

print(
    "Benchmark exact objective:",
    benchmark_exact_objective
)

print(
    "Final accepted exact objective:",
    final_exact_objective
)


,attempt,accepted,trust_scale,surrogate_objective,predicted_reduction,exact_objective,lowest_trial_objective,exact_improvement,alpha,normalized_model_mismatch,projected_command_change,termination_note
0,1,False,0.250000,88152.901772,185.848941,88338.750713,88368.675734,0.0,0.0,NaN,NaN,rejected_exact_replay
1,2,False,0.125000,88211.190324,127.560390,88338.750713,88354.469681,0.0,0.0,NaN,NaN,rejected_exact_replay
2,3,False,0.062500,88264.430942,74.319771,88338.750713,88346.687130,0.0,0.0,NaN,NaN,rejected_exact_replay
3,4,False,0.031250,88298.627905,40.122809,88338.750713,88342.730088,0.0,0.0,NaN,NaN,rejected_exact_replay
4,5,False,0.015625,88317.813636,20.937078,88338.750713,88340.686786,0.0,0.0,NaN,NaN,rejected_exact_replay
5,6,False,0.007812,88328.031862,10.718852,88338.750713,88339.699063,0.0,0.0,NaN,NaN,rejected_exact_replay


Sequential-QP termination status: no_exact_improving_step
Benchmark exact objective: 88338.75071339465
Final accepted exact objective: 88338.75071339465


In [12]:
# 8. Authoritative exact CTM replay of the final commands.
#
# The accepted trajectory already came from the exact simulator; this cell
# independently re-simulates the final command schedule and asserts equality.

qp_verification_history = (
    simulate_exact_command_matrix(
        final_commands
    )
)

verification_objective = float(
    sum(
        qp_verification_history[
            "total_objective"
        ]
    )
)

verification_state_error = float(
    np.max(
        np.abs(
            history_state_matrix(
                qp_verification_history
            )
            - nominal_states
        )
    )
)

if abs(
    verification_objective
    - final_exact_objective
) > 1e-8:
    raise AssertionError(
        "Final verification replay does not match "
        "the accepted exact objective."
    )

if verification_state_error > 1e-8:
    raise AssertionError(
        "Final verification replay does not match "
        "the accepted exact state trajectory."
    )

qp_exact_history = qp_verification_history

print(
    "PASS: final commands re-simulated; objective =",
    verification_objective
)


PASS: final commands re-simulated; objective = 88338.75071339465


In [13]:
# 9. Compare exact benchmark with exact QP replay.


def aggregate_exact_history(history):
    totals = {
        "mainline_tts": float(
            sum(history["mainline_tts"])
        ),

        "ramp_queue_tts": float(
            sum(history["ramp_queue_tts"])
        ),

        "upstream_boundary_tts": float(
            sum(
                history[
                    "upstream_boundary_tts"
                ]
            )
        ),

        "external_entry_tts": float(
            sum(
                history[
                    "external_entry_tts"
                ]
            )
        ),

        "fairness_penalty": float(
            sum(history["fairness_penalty"])
        ),

        "safe_exposure": float(
            sum(history["safe_exposure"])
        ),

        "spillback_exposure": float(
            sum(
                history[
                    "spillback_exposure"
                ]
            )
        ),

        "weighted_safe_penalty": float(
            sum(
                history[
                    "weighted_safe_penalty"
                ]
            )
        ),

        "weighted_spillback_penalty": float(
            sum(
                history[
                    "weighted_spillback_penalty"
                ]
            )
        ),

        "raw_objective": float(
            sum(history["total_objective"])
        ),
    }


    totals[
        "final_total_inventory"
    ] = (
        sum(
            float(history["x_final"][cell])
            for cell in cells
        )

        + sum(
            float(history["R_final"][ramp])
            + float(history["B_final"][ramp])
            for ramp in ramps
        )

        + float(
            history[
                "upstream_boundary_queue_final"
            ]
        )

        + sum(
            float(
                history[
                    "external_entry_queue_final"
                ][cell]
            )
            for cell in cells
        )
    )


    reconstructed_objective = (
        AW
        * totals["mainline_tts"]

        + totals["ramp_queue_tts"]

        + totals[
            "upstream_boundary_tts"
        ]

        + totals[
            "external_entry_tts"
        ]

        + totals[
            "fairness_penalty"
        ]

        + totals[
            "weighted_safe_penalty"
        ]

        + totals[
            "weighted_spillback_penalty"
        ]
    )


    if abs(
        totals["raw_objective"]
        - reconstructed_objective
    ) > 1e-7:
        raise AssertionError(
            "Exact replay objective does not equal "
            "the sum of its components."
        )


    return totals


qp_exact_totals = aggregate_exact_history(
    qp_exact_history
)


if termination_status in {
    "converged",
    "local_surrogate_stationary",
}:
    qp_policy_label = (
        "Sequential convex QP — exact replay"
    )
else:
    qp_policy_label = (
        "Best exact-replay SCP iterate — "
        f"{termination_status}"
    )


benchmark_final_inventory = float(
    S["official_service_metrics"][
        "final_total_system_inventory"
    ]
)


comparison = pd.DataFrame([
    {
        "policy": (
            "Fully open / no control"
        ),

        "mainline_tts": (
            benchmark_totals[
                "mainline_tts"
            ]
        ),

        "ramp_queue_tts": (
            benchmark_totals[
                "ramp_queue_tts"
            ]
        ),

        "upstream_boundary_tts": (
            benchmark_totals[
                "upstream_boundary_tts"
            ]
        ),

        "external_entry_tts": (
            benchmark_totals[
                "external_entry_tts"
            ]
        ),

        "spillback_exposure": (
            benchmark_totals[
                "spillback_exposure"
            ]
        ),

        "safe_exposure": (
            benchmark_totals[
                "safe_exposure"
            ]
        ),

        "fairness_penalty": (
            benchmark_totals[
                "fairness_penalty"
            ]
        ),

        "weighted_spillback_penalty": (
            benchmark_totals[
                "weighted_spillback_penalty"
            ]
        ),

        "weighted_safe_penalty": (
            benchmark_totals[
                "weighted_safe_penalty"
            ]
        ),

        "raw_objective": (
            benchmark_totals[
                "raw_objective"
            ]
        ),

        "final_total_inventory": (
            benchmark_final_inventory
        ),
    },

    {
        "policy": qp_policy_label,

        "mainline_tts": (
            qp_exact_totals[
                "mainline_tts"
            ]
        ),

        "ramp_queue_tts": (
            qp_exact_totals[
                "ramp_queue_tts"
            ]
        ),

        "upstream_boundary_tts": (
            qp_exact_totals[
                "upstream_boundary_tts"
            ]
        ),

        "external_entry_tts": (
            qp_exact_totals[
                "external_entry_tts"
            ]
        ),

        "spillback_exposure": (
            qp_exact_totals[
                "spillback_exposure"
            ]
        ),

        "safe_exposure": (
            qp_exact_totals[
                "safe_exposure"
            ]
        ),

        "fairness_penalty": (
            qp_exact_totals[
                "fairness_penalty"
            ]
        ),

        "weighted_spillback_penalty": (
            qp_exact_totals[
                "weighted_spillback_penalty"
            ]
        ),

        "weighted_safe_penalty": (
            qp_exact_totals[
                "weighted_safe_penalty"
            ]
        ),

        "raw_objective": (
            qp_exact_totals[
                "raw_objective"
            ]
        ),

        "final_total_inventory": (
            qp_exact_totals[
                "final_total_inventory"
            ]
        ),
    },
])


comparison[
    "objective_change_vs_benchmark"
] = (
    comparison["raw_objective"]
    - float(
        benchmark_totals[
            "raw_objective"
        ]
    )
)


comparison[
    "objective_percent_change"
] = (
    100.0
    * comparison[
        "objective_change_vs_benchmark"
    ]
    / float(
        benchmark_totals[
            "raw_objective"
        ]
    )
)


comparison_columns = [
    "policy",
    "mainline_tts",
    "ramp_queue_tts",
    "upstream_boundary_tts",
    "external_entry_tts",
    "spillback_exposure",
    "weighted_spillback_penalty",
    "safe_exposure",
    "weighted_safe_penalty",
    "fairness_penalty",
    "raw_objective",
    "final_total_inventory",
    "objective_change_vs_benchmark",
    "objective_percent_change",
]

display(
    comparison[
        comparison_columns
    ].round(6)
)


print(
    "Last local QP surrogate objective:",
    last_qp_result[
        "components"
    ]["raw_objective"]
)

print(
    "Authoritative exact-replay objective:",
    qp_exact_totals[
        "raw_objective"
    ]
)


,policy,mainline_tts,ramp_queue_tts,upstream_boundary_tts,external_entry_tts,spillback_exposure,weighted_spillback_penalty,safe_exposure,weighted_safe_penalty,fairness_penalty,raw_objective,final_total_inventory,objective_change_vs_benchmark,objective_percent_change
0,Fully open / no control,86893.445776,319.342487,0.0,0.0,0.0,0.0,2137.288992,1068.644496,57.317955,88338.750713,915.076524,0.0,0.0
1,Best exact-replay SCP iterate — no_exact_impro...,86893.445776,319.342487,0.0,0.0,0.0,0.0,2137.288992,1068.644496,57.317955,88338.750713,915.076524,0.0,0.0


Last local QP surrogate objective: 88328.03186167916
Authoritative exact-replay objective: 88338.75071339465


In [14]:
# 10. Mathematical invariant checks for the QP exact replay.

qp_invariants = {
    "mainline_mass_residual": (
        compute_mainline_mass_residual(
            qp_exact_history
        )
    ),

    "upstream_boundary_mass_residual": (
        compute_upstream_boundary_mass_residual(
            qp_exact_history
        )
    ),

    "external_entry_mass_residual": (
        compute_external_entry_mass_residual(
            qp_exact_history
        )
    ),

    "max_sending_violation": (
        compute_max_sending_violation(
            qp_exact_history
        )
    ),

    "max_receiving_violation": (
        compute_max_receiving_violation(
            qp_exact_history
        )
    ),

    "max_avila_merge_violation": (
        compute_max_merge_violation(
            qp_exact_history
        )
    ),
}


qp_ramp_mass_residual = (
    sum(
        float(S["ramp_queue_0"][ramp])
        + float(
            S["ramp_overflow_queue_0"][ramp]
        )
        for ramp in ramps
    )

    + sum(
        float(
            S["ramp_arrival_series"][ramp][step]
        )
        for ramp in ramps
        for step in range(T)
    )

    - sum(
        float(
            qp_exact_history[
                "actual_release"
            ][step][ramp]
        )
        for ramp in ramps
        for step in range(T)
    )

    - sum(
        float(
            qp_exact_history["R_final"][ramp]
        )
        + float(
            qp_exact_history["B_final"][ramp]
        )
        for ramp in ramps
    )
)


qp_invariants[
    "ramp_mass_residual"
] = qp_ramp_mass_residual


qp_invariants_df = pd.DataFrame([
    {
        "check": check,
        "value": value,
    }
    for check, value in qp_invariants.items()
])


display(
    qp_invariants_df.round(12)
)


for residual_name in [
    "mainline_mass_residual",
    "upstream_boundary_mass_residual",
    "external_entry_mass_residual",
    "ramp_mass_residual",
]:
    if abs(
        qp_invariants[residual_name]
    ) > 1e-8:
        raise AssertionError(
            f"{residual_name} failed."
        )


for violation_name in [
    "max_sending_violation",
    "max_receiving_violation",
    "max_avila_merge_violation",
]:
    if (
        qp_invariants[violation_name]
        > 1e-8
    ):
        raise AssertionError(
            f"{violation_name} failed."
        )


print(
    "PASS: QP commands were evaluated by the exact CTM "
    "and all physical invariants passed."
)


,check,value
0,mainline_mass_residual,4.000000e-12
1,upstream_boundary_mass_residual,0.000000e+00
2,external_entry_mass_residual,0.000000e+00
3,max_sending_violation,0.000000e+00
4,max_receiving_violation,0.000000e+00
5,max_avila_merge_violation,0.000000e+00
6,ramp_mass_residual,-0.000000e+00


PASS: QP commands were evaluated by the exact CTM and all physical invariants passed.
